# MRI → CT synthesis — pix2pix + PatchNCE

Training notebook for Kaggle. Before running:

1. **Add the dataset** — *Add Input* → your uploaded `mri-ct-paired-slices`.
2. **Turn on the GPU** — *Settings* → *Accelerator* → **GPU P100** or **T4 x2**.
3. **Turn on internet** if you clone the repo from GitHub (Settings → Internet).

**Sessions are time-limited and will be killed.** That is expected and handled:
every epoch writes a full checkpoint, and re-running this notebook with
`RESUME = 'auto'` picks up exactly where it stopped — same RNG stream, same
optimizer state, same LR schedule. Do **not** lower `n_epochs` to make a run
"fit" in one session: the LR schedule is defined against the total, so that
would change the learning rate of the epochs you already ran.

## 1. Get the code

In [ ]:
import os, sys, subprocess

# Option A — clone from GitHub (needs Internet enabled in Settings).
REPO_URL = 'https://github.com/Ayan2582/MRI_CT_preprocessing_pipeline.git'
REPO_DIR = '/kaggle/working/MRI_CT_preprocessing_pipeline'

# Option B — upload the `model/` folder as a second Kaggle dataset and point
# REPO_DIR at it instead. Nothing outside model/ is needed for training.

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)

In [ ]:
import torch
print('torch      ', torch.__version__)
print('cuda        ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device      ', torch.cuda.get_device_name(0))
    print('memory      ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

# PyYAML and pandas are preinstalled on Kaggle; nothing else is needed.
import yaml, pandas  # noqa: F401
print('deps ok')

## 2. Point the config at the mounted dataset

The manifest ships with **relative** paths, so only `data.root` changes between
this machine and Kaggle. Everything else in the config is identical, which is
what makes a Kaggle run comparable to a local one.

In [ ]:
import glob

candidates = sorted(glob.glob('/kaggle/input/*/manifest.csv'))
assert candidates, 'No dataset with a manifest.csv found. Add it via *Add Input*.'
DATA_ROOT = os.path.dirname(candidates[0])
print('dataset root:', DATA_ROOT)

import pandas as pd
mf = pd.read_csv(os.path.join(DATA_ROOT, 'manifest.csv'))
print(f'{len(mf)} pairs, {mf.subject_id.nunique()} subjects')
print(mf.body_region.value_counts().to_string())

In [ ]:
# ── Experiment selection ────────────────────────────────────────────────────
# The ladder, in the order it is meant to be run:
#   exp0_l1_only    is the GAN earning its keep at all?
#   exp1_pix2pix    the standard recipe, your baseline  <- START HERE
#   exp2_paper      the target loss at textbook weights
#   exp3_nce_heavy  lean on NCE - an open question, not a favourite
#   exp4_nce_max    where does hallucination start?
CONFIG = 'model/configs/exp1_pix2pix.yaml'

RESUME = 'auto'   # picks up this run's last.pt if one exists

OVERRIDES = [
    f'data.root={DATA_ROOT}',
    f'data.manifest={DATA_ROOT}/manifest.csv',
    f'data.splits={DATA_ROOT}/splits.json',
    'run.out_dir=/kaggle/working/runs',
    'train.batch_size=8',       # ~11 GB at 256px on a P100; drop to 4 if OOM
    'train.n_epochs=200',       # keep this at the TARGET, not what fits today
    'runtime.num_workers=2',    # Kaggle gives 2 vCPU on GPU instances
    'runtime.amp=true',
]

# Change nothing else. Any loss variation is a one-line override, e.g.
#   OVERRIDES += ['loss.lambda_nce=0']       -> plain pix2pix
#   OVERRIDES += ['loss.lambda_gan=0']       -> L1-only regression floor
print(CONFIG)
print('\n'.join('  ' + o for o in OVERRIDES))

## 3. Sanity check before spending GPU hours

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-7s %(message)s',
                    datefmt='%H:%M:%S', force=True)

from model.config import load_config
from model.data.manifest import load_manifest
from model.data.splits import load_split
from model.data.dataset import build_datasets

cfg = load_config(CONFIG, OVERRIDES)
manifest = load_manifest(cfg.data.manifest)
split = load_split(cfg.data.splits)
datasets = build_datasets(cfg, manifest, split)

item = datasets['train'][0]
print('A', tuple(item['A'].shape), 'B', tuple(item['B'].shape),
      'range', (float(item['A'].min()), float(item['A'].max())))
print('config hash', cfg['_hash'])
print({k: len(v.df) for k, v in datasets.items()})

## 4. Train

In [ ]:
from model.training.trainer import Trainer

trainer = Trainer(cfg, datasets)
trainer.maybe_resume(RESUME)
trainer.train()

## 5. Read the curves

`val/mae_norm` on the EMA generator is the number that ranks epochs. The GAN
losses are diagnostics, not quality measures — see `docs/gan_evaluation_guide.md`
for what each shape means.

In [ ]:
import matplotlib.pyplot as plt

hist = pd.read_csv(os.path.join(trainer.run_dir, 'metrics.csv'))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].plot(hist.epoch, hist['val/mae_norm'], label='val mae_norm')
best = hist['val/mae_norm'].idxmin()
ax[0].axvline(hist.epoch[best], ls='--', c='green',
              label=f'best @ {int(hist.epoch[best])}')
ax[0].set_title('SELECTION METRIC (lower is better)'); ax[0].legend()

for col in ('train/D_acc_real', 'train/D_acc_fake'):
    if col in hist:
        ax[1].plot(hist.epoch, hist[col], label=col.split('/')[1])
ax[1].axhspan(0.6, 0.85, alpha=0.12, color='green')
ax[1].axhline(0.5, ls=':', c='grey'); ax[1].set_ylim(0, 1.05)
ax[1].set_title('D accuracy — green band is healthy'); ax[1].legend()

for col in ('train/G_GAN', 'train/G_L1', 'train/G_NCE'):
    if col in hist:
        ax[2].plot(hist.epoch, hist[col], label=col.split('/')[1])
ax[2].set_yscale('log'); ax[2].set_title('loss terms (log)'); ax[2].legend()

for a in ax:
    a.set_xlabel('epoch'); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# The fixed sample panel: MRI | real CT | synth CT | error, same slices every time.
from IPython.display import Image, display

panels = sorted(glob.glob(os.path.join(trainer.sample_dir, '*.png')))
if panels:
    print(os.path.basename(panels[-1]))
    display(Image(panels[-1]))
else:
    print('no panels yet — they are written every logging.sample_every epochs')

## 6. Save the results

Everything under `/kaggle/working` is downloadable from the *Output* tab once
the session ends. Checkpoints are ~650 MB each, so only `best.pt` and `last.pt`
are kept — `last.pt` is the one to restore for the next session's resume.

In [ ]:
for path in sorted(glob.glob(os.path.join(trainer.run_dir, '**', '*'), recursive=True)):
    if os.path.isfile(path):
        print(f'{os.path.getsize(path) / 1e6:9.1f} MB  {os.path.relpath(path, trainer.run_dir)}')

print()
print('To continue in a new session: Save Version, then add this run as a')
print("Notebook Output input, restore last.pt, and re-run with RESUME='auto'.")

### Resuming — run this BEFORE the training cell in a follow-up session

Add the previous run as a *Notebook Output* input first, then uncomment.

In [ ]:
# import shutil
# RUN_NAME = 'exp1_pix2pix'
# prev = glob.glob(f'/kaggle/input/*/runs/{RUN_NAME}/checkpoints/last.pt')
# assert prev, 'previous checkpoint not found - check the Notebook Output input'
# dst = f'/kaggle/working/runs/{RUN_NAME}/checkpoints'
# os.makedirs(dst, exist_ok=True)
# shutil.copy(prev[0], os.path.join(dst, 'last.pt'))
# print('restored', prev[0])

## 7. Final evaluation — once, at the end

Run this on the **test** split only after the configuration is frozen. Every
time you read a test number and then change something in response, the test set
becomes a second validation set. Six subjects will not survive that repeatedly.

In [ ]:
# trainer.load_checkpoint(os.path.join(trainer.ckpt_dir, 'best.pt'))
# results = trainer.validate(trainer.start_epoch - 1, split='test')
# print(trainer.metrics.format_table(results))